In [1]:
!pip install -q transformers accelerate bitsandbytes requests

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import os
import gc 
from google.colab import drive

In [3]:
# 1. Setup
drive.mount('/content/drive')
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
def run_rq2_pipeline(persona_choice, model_id="deepseek-ai/deepseek-coder-7b-instruct-v1.5"):
    # --- 1. Path Configuration ---
    drive_root = '/content/drive/MyDrive/Project'
    input_path = os.path.join(drive_root, 'data/filtered_experimental_set.csv')
    output_dir = os.path.join(drive_root, 'results')
    prompt_path = os.path.join(drive_root, 'prompts', f'{persona_choice}_persona.txt')
    output_path = os.path.join(output_dir, f'rq2_{persona_choice}_results.csv')

    # --- 2. Resume Logic (Prevent Duplicates) ---
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        # Ensure we only have unique indices
        existing_df = existing_df.drop_duplicates(subset=['index'])
        processed_indices = set(existing_df['index'].astype(int).tolist())
        results_list = existing_df.to_dict('records')
        print(f"--- Resuming {persona_choice.upper()}: {len(processed_indices)} samples already found. ---")
    else:
        processed_indices = set()
        results_list = []
        print(f"--- Starting {persona_choice.upper()} from scratch. ---")

    # --- 3. Data & Prompt Loading ---
    df = pd.read_csv(input_path)
    # Ensure indices are handled as integers
    df['index'] = df['index'].astype(int)
    
    with open(prompt_path, 'r', encoding='utf-8') as f:
        system_instruction = f.read().strip()

    # --- 4. Model Loading (Only if work is remaining) ---
    if len(processed_indices) >= len(df):
        print("All samples already processed. Skipping model load.")
        return

    print(f"--- Loading Model: {model_id} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # 5. Inference Loop ---
    print(f"--- Running Inference ---")
    new_count = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        curr_idx = int(row['index'])
        if curr_idx in processed_indices:
            continue

        code = row['code']
        prompt = f"{system_instruction}\n\nCODE:\n{code}\n\nExplanation:"
        
        try:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2500).to("cuda")
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=400,         # Reduced to prevent synonym loops/timeouts
                    temperature=0.1,            # Stable, professional output
                    do_sample=True,
                    repetition_penalty=1.1,    
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
            
            explanation = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
            # Simple heuristic to check schema adherence
            # check if the required headers exist in the response
            required_headers = ['VULNERABILITY:', 'ANALYSIS:', 'IMPACT:']
            adheres_to_schema = all(header in explanation for header in required_headers)
            
            # Check for "filler" which was explicitly forbidden (Instructional Drift)
            has_filler = len(explanation.split('VULNERABILITY:')[0].strip()) > 0
            
            results_list.append({
                'index': curr_idx,
                'cwe': row.get('cwe', 'Unknown'),
                'persona': persona_choice,
                'generated_explanation': explanation.strip(),
                'adheres_to_schema': adheres_to_schema,
                'instructional_drift': has_filler  # True if model talked before the first header
            })
            
            new_count += 1
            
            # Save every 2 samples and clear cache
            if new_count % 2 == 0:
                pd.DataFrame(results_list).to_csv(output_path, index=False)
                torch.cuda.empty_cache()
                if new_count % 10 == 0:
                    print(f" [Heartbeat] Processed {new_count} new samples. VRAM cleared.")

        except Exception as e:
            print(f"\nError at index {curr_idx}: {e}")
            continue

    # 6. Final Save & Cleanup
    pd.DataFrame(results_list).to_csv(output_path, index=False)
    print(f"--- Success! Final results saved to: {output_path} ---")
    
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [5]:
if __name__ == "__main__":
    choice = input("Enter persona (naive/formal/expert): ").strip().lower()
    run_verification = input(f"Run inference for '{choice}'? (y/n): ")
    if run_verification.lower() == 'y':
        run_rq2_pipeline(persona_choice=choice)

--- Starting NAIVE from scratch. ---
--- Loading Model: deepseek-ai/deepseek-coder-7b-instruct-v1.5 ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

--- Running Inference ---


 16%|█▌        | 10/64 [05:11<28:43, 31.92s/it]

...

 [Heartbeat] Processed 50 new samples. VRAM cleared.


 94%|█████████▍| 60/64 [30:03<02:04, 31.14s/it]

 [Heartbeat] Processed 60 new samples. VRAM cleared.


100%|██████████| 64/64 [32:04<00:00, 30.07s/it]


--- Success! Final results saved to: /content/drive/MyDrive/Project/results/rq2_naive_results.csv ---
